# osmextract benchmark (companion to `benchmarks.ipynb`)

[osmextract](https://github.com/ropensci/osmextract) is an **R** package, so its parsing tasks run here (with the **R kernel**) rather than inside the Python `benchmarks.ipynb`. This notebook runs the same two parsing tasks on the same Helsinki-region PBF and writes `osmextract_results.csv` next to it; `benchmarks.ipynb` loads that file and merges the osmextract rows into its results table and charts.

**Run this with an R kernel** (see the installation snippet in `benchmarks.ipynb`). It is kept fair the same way as the Python tools: only the `oe_read()` call is timed (median over `REPEATS` runs), and `force_vectortranslate = TRUE` re-does the PBF → features work each run (the osmextract analogue of QuackOSM's `ignore_cache`) rather than reading a cached GeoPackage.

In [ ]:
# Same input as benchmarks.ipynb: the Helsinki-region extract pyrosm bundles.
url <- "https://gist.github.com/HTenkanen/02dcfce32d447e65024d93d39ddb1812/raw/5fe7ffb625f091591d8c29128a9e3b37870a5012/Helsinki_region.osm.pbf"
pbf <- file.path(tempdir(), "Helsinki_region.osm.pbf")
if (!file.exists(pbf)) {
    message("Downloading the Helsinki-region PBF ...")
    download.file(url, pbf, mode = "wb", quiet = TRUE)
}
suppressPackageStartupMessages(library(osmextract))
REPEATS <- 3L
cat("osmextract", as.character(packageVersion("osmextract")),
    "| sf", as.character(packageVersion("sf")), "\nPBF:", pbf, "\n")

In [ ]:
# Time oe_read() of one PBF layer, filtered to a tag, and report the median + count.
bench_oe <- function(task, layer, tagkey) {
    q <- sprintf('SELECT * FROM "%s" WHERE %s IS NOT NULL', layer, tagkey)
    tryCatch({
        times <- numeric(REPEATS); n <- 0L
        for (i in seq_len(REPEATS)) {
            el <- system.time({
                x <- oe_read(pbf, layer = layer, query = q,
                             force_vectortranslate = TRUE, quiet = TRUE)
            })[["elapsed"]]
            times[i] <- el; n <- nrow(x)
        }
        data.frame(task = task, tool = "osmextract",
                   seconds = round(median(times), 2), features = n,
                   status = "ok", stringsAsFactors = FALSE)
    }, error = function(e) {
        data.frame(task = task, tool = "osmextract", seconds = NA_real_,
                   features = NA_integer_,
                   status = paste("error:", conditionMessage(e)),
                   stringsAsFactors = FALSE)
    })
}

In [ ]:
# Buildings = the multipolygons layer; roads = the lines layer (GDAL's OSM driver).
results <- rbind(
    bench_oe("buildings", "multipolygons", "building"),
    bench_oe("roads",     "lines",         "highway")
)
print(results)
write.csv(results, "osmextract_results.csv", row.names = FALSE)
cat("\nWrote osmextract_results.csv -- now re-run the \"Results\" section of",
    "benchmarks.ipynb to include osmextract.\n")